# 1. Tasks (asyncio.create_task())
A Task is a wrapper around a coroutine that schedules it to run on the event loop as soon as possible. It runs in the background. You can hold a reference to the task to check if it's done, cancel it, or retrieve its result later.

* When to use: When you want to fire off a background job (like sending a welcome email after user registration) and continue executing the rest of your function immediately, without waiting for the job to finish.

# 2. Awaiting Multiple Things: asyncio.gather()
If you have a batch of independent I/O operations (e.g., fetching user data, fetching their recent orders, and fetching their recommendations) and you need all the results before proceeding, you use asyncio.gather().

* How it works: It takes a list of coroutines (or Tasks), schedules them concurrently, and waits for all of them to complete. It returns a list of results in the exact same order you passed them in.

* The Gotcha: If one coroutine inside gather() raises an exception, gather() immediately throws that exception to the caller. The other coroutines will continue running in the background, but you won't get their results via this gather call unless you handle the error.

# 3. Granular Control: asyncio.wait()
asyncio.wait() is the more flexible, lower-level cousin of gather(). Instead of returning a list of ordered results, it returns two sets of Tasks: those that are done and those that are pending.

* How it works: You pass it a set of Tasks (not raw coroutines) and specify a return_when condition.

* Conditions: You can tell it to return ALL_COMPLETED (default), FIRST_COMPLETED (useful if you are querying multiple redundant servers and just want the fastest reply), or FIRST_EXCEPTION.

# 4. Setting Strict Time Limits: asyncio.wait_for()
In backend engineering, you cannot let a slow external API hang your application indefinitely. asyncio.wait_for() wraps a coroutine and enforces a timeout.

* How it works: If the coroutine doesn't finish within the specified seconds, it raises an asyncio.TimeoutError and cancels the underlying coroutine.

In [2]:
import asyncio
import time

# ---------------------------------------------------------
# Simulated Network Calls
# ---------------------------------------------------------
async def fetch_user_profile():
    print("Profile: Fetching...")
    await asyncio.sleep(1)
    print("Profile: Done!")
    return {"name": "Shubham", "id": 42}

async def fetch_user_orders():
    print("Orders: Fetching...")
    await asyncio.sleep(2)
    print("Orders: Done!")
    return ["Laptop", "Coffee"]

async def unstable_api_call():
    print("API: Calling unstable third-party service...")
    await asyncio.sleep(5)  # Takes way too long
    return "Success"

async def background_log(message):
    print(f"Background: Logging '{message}' to external service...")
    await asyncio.sleep(3)
    print(f"Background: Log '{message}' completed.")

# ---------------------------------------------------------
# The Orchestrator
# ---------------------------------------------------------
async def main():
    start_time = time.time()
    
    # --- 1. create_task() ---
    # We want to log the start of the process, but we don't want to wait 3 seconds 
    # for the log to finish before doing real work. We fire it and forget it (mostly).
    log_task = asyncio.create_task(background_log("System Boot"))
    
    # --- 2. gather() ---
    # We need both profile and orders to render a dashboard. 
    # We fetch them concurrently. It will take ~2 seconds total, not 3.
    print("\n--- Starting Gather ---")
    results = await asyncio.gather(
        fetch_user_profile(),
        fetch_user_orders()
    )
    profile_data = results[0]
    order_data = results[1]
    print(f"Gather Results: {profile_data}, {order_data}")
    
    # --- 3. wait_for() ---
    # We call a slow API but refuse to wait longer than 2 seconds.
    print("\n--- Starting Wait_For ---")
    try:
        api_result = await asyncio.wait_for(unstable_api_call(), timeout=2.0)
        print(f"API Result: {api_result}")
    except asyncio.TimeoutError:
        print("API Call: TimeoutError! The service was too slow, request cancelled.")
    
    # --- 4. wait() with FIRST_COMPLETED ---
    # Let's race two tasks.
    print("\n--- Starting Wait (First Completed) ---")
    task1 = asyncio.create_task(asyncio.sleep(1, result="Fast Runner"))
    task2 = asyncio.create_task(asyncio.sleep(3, result="Slow Runner"))
    
    done, pending = await asyncio.wait(
        [task1, task2], 
        return_when=asyncio.FIRST_COMPLETED
    )
    
    # 'done' is a set. We pop the finished task to get its result.
    winner = done.pop()
    print(f"Race Winner: {winner.result()}")
    
    # It's good practice to cancel pending tasks if we no longer need them
    for t in pending:
        t.cancel()

    print(f"\nTotal Execution Time: {time.time() - start_time:.2f}s")
    
    # Before we exit, let's wait for our background logger to finish so the 
    # script doesn't terminate while it's still running.
    await log_task

if __name__ == "__main__":
    # asyncio.run(main())
    await main()


--- Starting Gather ---
Background: Logging 'System Boot' to external service...
Profile: Fetching...
Orders: Fetching...
Profile: Done!
Orders: Done!
Gather Results: {'name': 'Shubham', 'id': 42}, ['Laptop', 'Coffee']

--- Starting Wait_For ---
API: Calling unstable third-party service...
Background: Log 'System Boot' completed.
API Call: TimeoutError! The service was too slow, request cancelled.

--- Starting Wait (First Completed) ---
Race Winner: Fast Runner

Total Execution Time: 5.04s
